# Exercice 09 - Nettoyage des donnees (Silver)

## Objectifs
- Comprendre la couche Silver du Data Lake
- Gerer les valeurs nulles
- Supprimer les doublons
- Standardiser les formats de donnees
- Valider la qualite des donnees

---

## 1. La couche Silver

```
+------------------+     +------------------+     +------------------+
|      BRONZE      |     |      SILVER      |     |       GOLD       |
+------------------+     +------------------+     +------------------+
|                  |     |                  |     |                  |
| Donnees brutes   | --> | Donnees propres  | --> | Donnees business |
| Non validees     |     | Validees         |     | Agregatees       |
| Avec doublons    |     | Sans doublons    |     | Optimisees       |
| Formats varies   |     | Formats standards|     | Pretes a l'usage |
|                  |     |                  |     |                  |
+------------------+     +------------------+     +------------------+

Nettoyages Silver :
- Suppression des doublons
- Gestion des valeurs nulles
- Standardisation des formats
- Validation des types
- Correction des erreurs evidentes
```

## 2. Configuration

In [52]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, IntegerType, DoubleType, DateType
from datetime import datetime

# Creer la SparkSession
spark = SparkSession.builder \
    .appName("Nettoyage Silver") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.4.1,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

date_traitement = datetime.now().strftime("%Y-%m-%d")
print(f"Spark pret - Date : {date_traitement}")

Spark pret - Date : 2026-03-25


## 3. Creer des donnees de test avec des problemes

In [53]:
# Donnees avec des problemes courants
data_problemes = [
    (1, "Alice", "alice@email.com", "Paris", 25, 50000.0),
    (2, "Bob", "BOB@EMAIL.COM", "paris", 30, 60000.0),      # Email en majuscules, ville minuscule
    (3, "Charlie", None, "Lyon", 35, None),                  # Valeurs nulles
    (4, "Diana", "diana@email.com", "  Marseille  ", -5, 70000.0),  # Espaces, age negatif
    (1, "Alice", "alice@email.com", "Paris", 25, 50000.0),   # Doublon
    (5, "Eve", "eve@email", "Toulouse", 28, 55000.0),        # Email invalide
    (6, "Frank", "frank@email.com", "", 40, 80000.0),        # Ville vide
    (7, "Grace", "grace@email.com", "Nice", 150, 65000.0),   # Age impossible
]

colonnes = ["id", "nom", "email", "ville", "age", "salaire"]
df_bronze = spark.createDataFrame(data_problemes, colonnes)

print("Donnees brutes (Bronze) :")
df_bronze.show(truncate=False)

Donnees brutes (Bronze) :
+---+-------+---------------+-------------+---+-------+
|id |nom    |email          |ville        |age|salaire|
+---+-------+---------------+-------------+---+-------+
|1  |Alice  |alice@email.com|Paris        |25 |50000.0|
|2  |Bob    |BOB@EMAIL.COM  |paris        |30 |60000.0|
|3  |Charlie|NULL           |Lyon         |35 |NULL   |
|4  |Diana  |diana@email.com|  Marseille  |-5 |70000.0|
|1  |Alice  |alice@email.com|Paris        |25 |50000.0|
|5  |Eve    |eve@email      |Toulouse     |28 |55000.0|
|6  |Frank  |frank@email.com|             |40 |80000.0|
|7  |Grace  |grace@email.com|Nice         |150|65000.0|
+---+-------+---------------+-------------+---+-------+



## 4. Supprimer les doublons

In [54]:
# Compter avant
print(f"Lignes avant : {df_bronze.count()}")

# Supprimer les doublons exacts
df_sans_doublons = df_bronze.dropDuplicates()
print(f"Lignes apres dropDuplicates() : {df_sans_doublons.count()}")

# Supprimer les doublons sur certaines colonnes
df_sans_doublons_id = df_bronze.dropDuplicates(["id"])
print(f"Lignes apres dropDuplicates(['id']) : {df_sans_doublons_id.count()}")

Lignes avant : 8
Lignes apres dropDuplicates() : 7
Lignes apres dropDuplicates(['id']) : 7


In [55]:
# Continuer avec les donnees sans doublons
df = df_sans_doublons
df.show()

+---+-------+---------------+-------------+---+-------+
| id|    nom|          email|        ville|age|salaire|
+---+-------+---------------+-------------+---+-------+
|  1|  Alice|alice@email.com|        Paris| 25|50000.0|
|  2|    Bob|  BOB@EMAIL.COM|        paris| 30|60000.0|
|  4|  Diana|diana@email.com|  Marseille  | -5|70000.0|
|  5|    Eve|      eve@email|     Toulouse| 28|55000.0|
|  6|  Frank|frank@email.com|             | 40|80000.0|
|  7|  Grace|grace@email.com|         Nice|150|65000.0|
|  3|Charlie|           NULL|         Lyon| 35|   NULL|
+---+-------+---------------+-------------+---+-------+



## 5. Gerer les valeurs nulles

In [56]:
# Compter les valeurs nulles par colonne
print("Valeurs nulles par colonne :")
for col in df.columns:
    nb_nulls = df.filter(F.col(col).isNull()).count()
    print(f"  {col}: {nb_nulls}")

Valeurs nulles par colonne :
  id: 0
  nom: 0
  email: 1
  ville: 0
  age: 0
  salaire: 1


In [59]:
# Option 1 : Supprimer les lignes avec des nulls
df_drop_na = df.dropna()
print(f"Apres dropna() : {df_drop_na.count()} lignes")
df.show()
df_drop_na.show()

Apres dropna() : 6 lignes
+---+-------+---------------+-------------+---+-------+
| id|    nom|          email|        ville|age|salaire|
+---+-------+---------------+-------------+---+-------+
|  1|  Alice|alice@email.com|        Paris| 25|50000.0|
|  2|    Bob|  BOB@EMAIL.COM|        paris| 30|60000.0|
|  4|  Diana|diana@email.com|  Marseille  | -5|70000.0|
|  5|    Eve|      eve@email|     Toulouse| 28|55000.0|
|  6|  Frank|frank@email.com|             | 40|80000.0|
|  7|  Grace|grace@email.com|         Nice|150|65000.0|
|  3|Charlie|           NULL|         Lyon| 35|   NULL|
+---+-------+---------------+-------------+---+-------+

+---+-----+---------------+-------------+---+-------+
| id|  nom|          email|        ville|age|salaire|
+---+-----+---------------+-------------+---+-------+
|  1|Alice|alice@email.com|        Paris| 25|50000.0|
|  2|  Bob|  BOB@EMAIL.COM|        paris| 30|60000.0|
|  4|Diana|diana@email.com|  Marseille  | -5|70000.0|
|  5|  Eve|      eve@email|     T

In [60]:
# Option 2 : Remplacer les nulls par des valeurs par defaut
df_fill = df.fillna({
    "email": "inconnu@example.com",
    "salaire": 0.0
})

df_fill.show()

+---+-------+-------------------+-------------+---+-------+
| id|    nom|              email|        ville|age|salaire|
+---+-------+-------------------+-------------+---+-------+
|  1|  Alice|    alice@email.com|        Paris| 25|50000.0|
|  2|    Bob|      BOB@EMAIL.COM|        paris| 30|60000.0|
|  4|  Diana|    diana@email.com|  Marseille  | -5|70000.0|
|  5|    Eve|          eve@email|     Toulouse| 28|55000.0|
|  6|  Frank|    frank@email.com|             | 40|80000.0|
|  7|  Grace|    grace@email.com|         Nice|150|65000.0|
|  3|Charlie|inconnu@example.com|         Lyon| 35|    0.0|
+---+-------+-------------------+-------------+---+-------+



In [62]:
# Option 3 : Remplacer par la moyenne (pour les numeriques)
moyenne_salaire = df.agg(F.avg("salaire")).collect()[0][0]
moyenne_salaire = round(moyenne_salaire, 2)
print(f"Salaire moyen : {moyenne_salaire}")

df_fill_avg = df.fillna({"salaire": moyenne_salaire})
df_fill_avg.show()

Salaire moyen : 63333.33
+---+-------+---------------+-------------+---+--------+
| id|    nom|          email|        ville|age| salaire|
+---+-------+---------------+-------------+---+--------+
|  1|  Alice|alice@email.com|        Paris| 25| 50000.0|
|  2|    Bob|  BOB@EMAIL.COM|        paris| 30| 60000.0|
|  4|  Diana|diana@email.com|  Marseille  | -5| 70000.0|
|  5|    Eve|      eve@email|     Toulouse| 28| 55000.0|
|  6|  Frank|frank@email.com|             | 40| 80000.0|
|  7|  Grace|grace@email.com|         Nice|150| 65000.0|
|  3|Charlie|           NULL|         Lyon| 35|63333.33|
+---+-------+---------------+-------------+---+--------+



## 6. Standardiser les formats

In [63]:
# Appliquer plusieurs nettoyages
df_clean = df_fill \
    .withColumn("email", F.lower(F.col("email"))) \
    .withColumn("ville", F.trim(F.col("ville"))) \
    .withColumn("ville", F.initcap(F.col("ville"))) \
    .withColumn("nom", F.initcap(F.col("nom")))

print("Apres standardisation :")
df_clean.show(truncate=False)

Apres standardisation :
+---+-------+-------------------+---------+---+-------+
|id |nom    |email              |ville    |age|salaire|
+---+-------+-------------------+---------+---+-------+
|1  |Alice  |alice@email.com    |Paris    |25 |50000.0|
|2  |Bob    |bob@email.com      |Paris    |30 |60000.0|
|4  |Diana  |diana@email.com    |Marseille|-5 |70000.0|
|5  |Eve    |eve@email          |Toulouse |28 |55000.0|
|6  |Frank  |frank@email.com    |         |40 |80000.0|
|7  |Grace  |grace@email.com    |Nice     |150|65000.0|
|3  |Charlie|inconnu@example.com|Lyon     |35 |0.0    |
+---+-------+-------------------+---------+---+-------+



In [64]:
# Traiter les villes vides
df_clean = df_clean.withColumn(
    "ville",
    F.when(F.col("ville") == "", "Inconnue").otherwise(F.col("ville"))
)

df_clean.show(truncate=False)

+---+-------+-------------------+---------+---+-------+
|id |nom    |email              |ville    |age|salaire|
+---+-------+-------------------+---------+---+-------+
|1  |Alice  |alice@email.com    |Paris    |25 |50000.0|
|2  |Bob    |bob@email.com      |Paris    |30 |60000.0|
|4  |Diana  |diana@email.com    |Marseille|-5 |70000.0|
|5  |Eve    |eve@email          |Toulouse |28 |55000.0|
|6  |Frank  |frank@email.com    |Inconnue |40 |80000.0|
|7  |Grace  |grace@email.com    |Nice     |150|65000.0|
|3  |Charlie|inconnu@example.com|Lyon     |35 |0.0    |
+---+-------+-------------------+---------+---+-------+



## 7. Valider les donnees

In [65]:
# Valider l'age (entre 0 et 120)
df_valid = df_clean.withColumn(
    "age_valide",
    F.when((F.col("age") >= 0) & (F.col("age") <= 120), True).otherwise(False)
)

print("Ages invalides :")
df_valid.filter(F.col("age_valide") == False).show()

Ages invalides :
+---+-----+---------------+---------+---+-------+----------+
| id|  nom|          email|    ville|age|salaire|age_valide|
+---+-----+---------------+---------+---+-------+----------+
|  4|Diana|diana@email.com|Marseille| -5|70000.0|     false|
|  7|Grace|grace@email.com|     Nice|150|65000.0|     false|
+---+-----+---------------+---------+---+-------+----------+



In [66]:
# Corriger ou filtrer les ages invalides
df_age_corrige = df_clean.withColumn(
    "age",
    F.when(F.col("age") < 0, None)
     .when(F.col("age") > 120, None)
     .otherwise(F.col("age"))
)

df_age_corrige.show()

+---+-------+-------------------+---------+----+-------+
| id|    nom|              email|    ville| age|salaire|
+---+-------+-------------------+---------+----+-------+
|  1|  Alice|    alice@email.com|    Paris|  25|50000.0|
|  2|    Bob|      bob@email.com|    Paris|  30|60000.0|
|  4|  Diana|    diana@email.com|Marseille|NULL|70000.0|
|  5|    Eve|          eve@email| Toulouse|  28|55000.0|
|  6|  Frank|    frank@email.com| Inconnue|  40|80000.0|
|  7|  Grace|    grace@email.com|     Nice|NULL|65000.0|
|  3|Charlie|inconnu@example.com|     Lyon|  35|    0.0|
+---+-------+-------------------+---------+----+-------+



In [67]:
# Valider le format email avec regex
pattern_email = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'

df_email_valide = df_age_corrige.withColumn(
    "email_valide",
    F.col("email").rlike(pattern_email)
)

print("Emails invalides :")
df_email_valide.filter(F.col("email_valide") == False).show(truncate=False)

Emails invalides :
+---+---+---------+--------+---+-------+------------+
|id |nom|email    |ville   |age|salaire|email_valide|
+---+---+---------+--------+---+-------+------------+
|5  |Eve|eve@email|Toulouse|28 |55000.0|false       |
+---+---+---------+--------+---+-------+------------+



## 8. Pipeline de nettoyage complet

In [68]:
def nettoyer_donnees(df):
    """
    Pipeline de nettoyage complet.
    
    Args:
        df: DataFrame brut (Bronze)
    
    Returns:
        DataFrame nettoye (Silver)
    """
    # 1. Supprimer les doublons
    df = df.dropDuplicates(["id"])
    
    # 2. Gerer les nulls
    df = df.fillna({
        "email": "inconnu@example.com",
        "ville": "Inconnue",
        "salaire": 0.0
    })
    
    # 3. Standardiser les formats
    df = df.withColumn("email", F.lower(F.trim(F.col("email")))) \
           .withColumn("ville", F.initcap(F.trim(F.col("ville")))) \
           .withColumn("nom", F.initcap(F.trim(F.col("nom"))))
    
    # 4. Traiter les villes vides
    df = df.withColumn(
        "ville",
        F.when(F.col("ville") == "", "Inconnue").otherwise(F.col("ville"))
    )
    
    # 5. Valider et corriger l'age
    df = df.withColumn(
        "age",
        F.when((F.col("age") >= 0) & (F.col("age") <= 120), F.col("age"))
         .otherwise(None)
    )
    
    # 6. Ajouter les metadonnees
    df = df.withColumn("_cleaned_date", F.lit(datetime.now().strftime("%Y-%m-%d %H:%M:%S")))
    
    return df

In [69]:
# Appliquer le pipeline
df_silver = nettoyer_donnees(df_bronze)

print("Donnees nettoyees (Silver) :")
df_silver.show(truncate=False)

Donnees nettoyees (Silver) :
+---+-------+-------------------+---------+----+-------+-------------------+
|id |nom    |email              |ville    |age |salaire|_cleaned_date      |
+---+-------+-------------------+---------+----+-------+-------------------+
|1  |Alice  |alice@email.com    |Paris    |25  |50000.0|2026-03-25 14:17:47|
|2  |Bob    |bob@email.com      |Paris    |30  |60000.0|2026-03-25 14:17:47|
|3  |Charlie|inconnu@example.com|Lyon     |35  |0.0    |2026-03-25 14:17:47|
|4  |Diana  |diana@email.com    |Marseille|NULL|70000.0|2026-03-25 14:17:47|
|5  |Eve    |eve@email          |Toulouse |28  |55000.0|2026-03-25 14:17:47|
|6  |Frank  |frank@email.com    |Inconnue |40  |80000.0|2026-03-25 14:17:47|
|7  |Grace  |grace@email.com    |Nice     |NULL|65000.0|2026-03-25 14:17:47|
+---+-------+-------------------+---------+----+-------+-------------------+



## 9. Rapport de qualite

In [70]:
def rapport_qualite(df_avant, df_apres):
    """
    Genere un rapport de qualite des donnees.
    """
    print("=" * 50)
    print("RAPPORT DE QUALITE")
    print("=" * 50)
    
    print(f"\nLignes avant  : {df_avant.count()}")
    print(f"Lignes apres  : {df_apres.count()}")
    print(f"Lignes retirees: {df_avant.count() - df_apres.count()}")
    
    print("\nValeurs nulles (apres nettoyage) :")
    for col in df_apres.columns:
        if not col.startswith("_"):
            nb_nulls = df_apres.filter(F.col(col).isNull()).count()
            pct = (nb_nulls / df_apres.count()) * 100 if df_apres.count() > 0 else 0
            print(f"  {col}: {nb_nulls} ({pct:.1f}%)")
    
    print("=" * 50)

rapport_qualite(df_bronze, df_silver)

RAPPORT DE QUALITE

Lignes avant  : 8
Lignes apres  : 7
Lignes retirees: 1

Valeurs nulles (apres nettoyage) :
  id: 0 (0.0%)
  nom: 0 (0.0%)
  email: 0 (0.0%)
  ville: 0 (0.0%)
  age: 2 (28.6%)
  salaire: 0 (0.0%)


## 10. Sauvegarder dans Silver

In [71]:
# Sauvegarder les donnees nettoyees
chemin_silver = f"s3a://silver/users/{date_traitement}"
df_silver.write.mode("overwrite").parquet(chemin_silver)

print(f"Sauvegarde Silver : {chemin_silver}")

Sauvegarde Silver : s3a://silver/users/2026-03-25


---

## Exercice

**Objectif** : Nettoyer les donnees Northwind customers

**Consigne** :
1. Lisez la table `customers` depuis PostgreSQL ou Bronze
2. Appliquez les nettoyages suivants :
   - Standardisez les noms de pays (majuscules)
   - Supprimez les espaces en debut/fin de texte
   - Gerez les valeurs nulles
3. Sauvegardez dans Silver

A vous de jouer :

In [72]:
# TODO: Lire les customers depuis PostgreSQL
jdbc_url = "jdbc:postgresql://postgres:5432/app"
jdbc_properties = {
    "user": "postgres",
    "password": "postgres",
    "driver": "org.postgresql.Driver"
}

df_customers = spark.read.jdbc(url=jdbc_url, table="customers", properties=jdbc_properties)
df_customers.show()

+-----------+--------------------+------------------+--------------------+--------------------+------------+------+-----------+-----------+--------------+--------------+
|customer_id|        company_name|      contact_name|       contact_title|             address|        city|region|postal_code|    country|         phone|           fax|
+-----------+--------------------+------------------+--------------------+--------------------+------------+------+-----------+-----------+--------------+--------------+
|      ALFKI| Alfreds Futterkiste|      Maria Anders|Sales Representative|       Obere Str. 57|      Berlin|  NULL|      12209|    Germany|   030-0074321|   030-0076545|
|      ANATR|Ana Trujillo Empa...|      Ana Trujillo|               Owner|Avda. de la Const...| México D.F.|  NULL|      05021|     Mexico|  (5) 555-4729|  (5) 555-3745|
|      ANTON|Antonio Moreno Ta...|    Antonio Moreno|               Owner|     Mataderos  2312| México D.F.|  NULL|      05023|     Mexico|  (5) 555-3

In [76]:
df_customers.select('address').show(truncate=False)

+-----------------------------+
|address                      |
+-----------------------------+
|Obere Str. 57                |
|Avda. de la Constitución 2222|
|Mataderos  2312              |
|120 Hanover Sq.              |
|Berguvsvägen  8              |
|Forsterstr. 57               |
|24, place Kléber             |
|C/ Araquil, 67               |
|12, rue des Bouchers         |
|23 Tsawassen Blvd.           |
|Fauntleroy Circus            |
|Cerrito 333                  |
|Sierras de Granada 9993      |
|Hauptstr. 29                 |
|Av. dos Lusíadas, 23         |
|Berkeley Gardens 12  Brewery |
|Walserweg 21                 |
|67, rue des Cinquante Otages |
|35 King George               |
|Kirchgasse 6                 |
+-----------------------------+
only showing top 20 rows


In [78]:
# TODO: Appliquer les nettoyages
# place the number at the beginning and the street name after
df_customers = df_customers.withColumn(
    "address",
    F.concat(
        F.regexp_extract(F.col("address"), r'(\d+)', 1),  # Extract first number
        F.lit(" "),
        F.trim(F.regexp_replace(F.col("address"), r'\d+', ''))  # Remove ALL numbers
    )
)

df_customers.select('address').show(truncate=False)

+-----------------------------+
|address                      |
+-----------------------------+
|57 Obere Str.                |
|2222 Avda. de la Constitución|
|2312 Mataderos               |
|120 Hanover Sq.              |
|8 Berguvsvägen               |
|57 Forsterstr.               |
|24 , place Kléber            |
|67 C/ Araquil,               |
|12 , rue des Bouchers        |
|23 Tsawassen Blvd.           |
| Fauntleroy Circus           |
|333 Cerrito                  |
|9993 Sierras de Granada      |
|29 Hauptstr.                 |
|23 Av. dos Lusíadas,         |
|12 Berkeley Gardens   Brewery|
|21 Walserweg                 |
|67 , rue des Cinquante Otages|
|35 King George               |
|6 Kirchgasse                 |
+-----------------------------+
only showing top 20 rows


In [87]:
#count the doublons and null values 
print(df_customers.count())
df_customers.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_customers.columns]).show()

91
+-----------+------------+------------+-------------+-------+----+-----------+-------+-----+---+
|customer_id|company_name|contact_name|contact_title|address|city|postal_code|country|phone|fax|
+-----------+------------+------------+-------------+-------+----+-----------+-------+-----+---+
|          0|           0|           0|            0|      0|   0|          1|      0|    0|  0|
+-----------+------------+------------+-------------+-------+----+-----------+-------+-----+---+



In [84]:
#drop the region 
df_customers = df_customers.drop("region")
df_customers.show()

+-----------+--------------------+------------------+--------------------+--------------------+------------+-----------+-----------+--------------+--------------+
|customer_id|        company_name|      contact_name|       contact_title|             address|        city|postal_code|    country|         phone|           fax|
+-----------+--------------------+------------------+--------------------+--------------------+------------+-----------+-----------+--------------+--------------+
|      ALFKI| Alfreds Futterkiste|      Maria Anders|Sales Representative|       57 Obere Str.|      Berlin|      12209|    Germany|   030-0074321|   030-0076545|
|      ANATR|Ana Trujillo Empa...|      Ana Trujillo|               Owner|2222 Avda. de la ...| México D.F.|      05021|     Mexico|  (5) 555-4729|  (5) 555-3745|
|      ANTON|Antonio Moreno Ta...|    Antonio Moreno|               Owner|      2312 Mataderos| México D.F.|      05023|     Mexico|  (5) 555-3932|          NULL|
|      AROUT|     Arou

In [86]:
#filling the null fax value with N/A
df_customers = df_customers.fillna({"fax": "N/A"})
df_customers.show()

+-----------+--------------------+------------------+--------------------+--------------------+------------+-----------+-----------+--------------+--------------+
|customer_id|        company_name|      contact_name|       contact_title|             address|        city|postal_code|    country|         phone|           fax|
+-----------+--------------------+------------------+--------------------+--------------------+------------+-----------+-----------+--------------+--------------+
|      ALFKI| Alfreds Futterkiste|      Maria Anders|Sales Representative|       57 Obere Str.|      Berlin|      12209|    Germany|   030-0074321|   030-0076545|
|      ANATR|Ana Trujillo Empa...|      Ana Trujillo|               Owner|2222 Avda. de la ...| México D.F.|      05021|     Mexico|  (5) 555-4729|  (5) 555-3745|
|      ANTON|Antonio Moreno Ta...|    Antonio Moreno|               Owner|      2312 Mataderos| México D.F.|      05023|     Mexico|  (5) 555-3932|           N/A|
|      AROUT|     Arou

In [91]:
#filling the null zip code with Inconnue
df_customers = df_customers.fillna({"postal_code": "Inconnue"})
df_customers.show()

+-----------+--------------------+------------------+--------------------+--------------------+------------+-----------+-----------+--------------+--------------+
|customer_id|        company_name|      contact_name|       contact_title|             address|        city|postal_code|    country|         phone|           fax|
+-----------+--------------------+------------------+--------------------+--------------------+------------+-----------+-----------+--------------+--------------+
|      ALFKI| Alfreds Futterkiste|      Maria Anders|Sales Representative|       57 Obere Str.|      Berlin|      12209|    Germany|   030-0074321|   030-0076545|
|      ANATR|Ana Trujillo Empa...|      Ana Trujillo|               Owner|2222 Avda. de la ...| México D.F.|      05021|     Mexico|  (5) 555-4729|  (5) 555-3745|
|      ANTON|Antonio Moreno Ta...|    Antonio Moreno|               Owner|      2312 Mataderos| México D.F.|      05023|     Mexico|  (5) 555-3932|           N/A|
|      AROUT|     Arou

In [93]:
# TODO: Sauvegarder dans Silver

df_customers.withColumn("date_traitement", F.lit(date_traitement)).show()
df_customers.withColumn("_endpoint", F.lit("jdbc:postgresql://postgres:5432/app"))
df_customers.withColumn('date_ingestion', F.lit(datetime.now().strftime("%Y-%m-%d %H:%M:%S")))
chemin_silver_customers = f"s3a://silver/customers/{date_traitement}"
df_customers.write.mode("overwrite").parquet(chemin_silver_customers)   
print(f"Sauvegarde Silver : {chemin_silver_customers}")


+-----------+--------------------+------------------+--------------------+--------------------+------------+-----------+-----------+--------------+--------------+---------------+
|customer_id|        company_name|      contact_name|       contact_title|             address|        city|postal_code|    country|         phone|           fax|date_traitement|
+-----------+--------------------+------------------+--------------------+--------------------+------------+-----------+-----------+--------------+--------------+---------------+
|      ALFKI| Alfreds Futterkiste|      Maria Anders|Sales Representative|       57 Obere Str.|      Berlin|      12209|    Germany|   030-0074321|   030-0076545|     2026-03-25|
|      ANATR|Ana Trujillo Empa...|      Ana Trujillo|               Owner|2222 Avda. de la ...| México D.F.|      05021|     Mexico|  (5) 555-4729|  (5) 555-3745|     2026-03-25|
|      ANTON|Antonio Moreno Ta...|    Antonio Moreno|               Owner|      2312 Mataderos| México D.

---

## Resume

Dans ce notebook, vous avez appris :
- Le role de la couche **Silver** dans un Data Lake
- Comment **supprimer les doublons** avec dropDuplicates()
- Comment **gerer les valeurs nulles** avec dropna() et fillna()
- Comment **standardiser les formats** (trim, lower, initcap)
- Comment **valider les donnees** avec des regles metier
- Comment creer un **pipeline de nettoyage** reutilisable

### Prochaine etape
Dans le prochain notebook, nous apprendrons les transformations avancees pour la couche Silver.